# **PREDICT SALES PRICE NOTEBOOK**

<br>

## Objectives

* Fit and evaluate a regression model to predict summed sales price of the 4 inherited houses.
* Fit and evaluate a regression model to predict sale price of any house in Ames, Iowa.
  
## Inputs

* outputs/datasets/collection/house_prices_after_inspection.csv
* Instructions on which variables to use for data cleaning and feature engineering (found in their respective notebooks).

## Outputs

* Train set (features and target)
* Test set (features and target)
* ML pipeline to sales price
* Feature Importance Plot

---

# Change working directory to the parent folder

Access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

Make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

# Load the Data

In [ ]:
import pandas as pd
df = (pd.read_csv(
  "outputs/datasets/collection/house_prices_after_inspection.csv").
  drop(labels=['EnclosedPorch', 'WoodDeckSF'], axis=1)
  )

df.tail()

* Command to render the plots directly within the output cells.

In [ ]:
%matplotlib inline

* To maintain a clean console, specific FutureWarnings are suppressed. I acknowledge, however, that these warnings often highlight critical information about future library updates and potential breaking changes.

In [ ]:
import warnings

# Suppress FutureWarnings from feature_engine
warnings.filterwarnings("ignore", category=FutureWarning, module='feature_engine')

# Suppress FutureWarnings from xgboost
warnings.filterwarnings("ignore", category=FutureWarning, module='xgboost')

# ML Pipeline: Regressor

### Step 1: Create ML pipeline

In [ ]:
from sklearn.pipeline import Pipeline

# Data Cleaning
from feature_engine.imputation import CategoricalImputer
from feature_engine.imputation import MeanMedianImputer

# Feature Engineering
from feature_engine.outliers import Winsorizer
from feature_engine.encoding import OrdinalEncoder
from feature_engine import transformation as vt
from feature_engine.selection import SmartCorrelatedSelection

# Feat Scaling
from sklearn.preprocessing import StandardScaler

# Feat Selection
from sklearn.feature_selection import SelectFromModel

# ML algorithms
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor


def PipelineOptimization(model):
    pipeline_base = Pipeline([

        ("categorical_imputation", CategoricalImputer(imputation_method='missing',
                                                  fill_value='Missing',
                                                  variables=['BsmtExposure', 
                                                             'BsmtFinType1', 
                                                             'GarageFinish', 
                                                            'KitchenQual'])),
        
        ("numerical_imputation", MeanMedianImputer(imputation_method='median',
                            variables= ['1stFlrSF', '2ndFlrSF', 'BedroomAbvGr',
                                         'BsmtFinSF1', 'BsmtUnfSF', 'GarageArea'
                                         , 'GarageYrBlt', 'GrLivArea', 'LotArea'
                                         , 'LotFrontage', 'MasVnrArea', 
                                         'OpenPorchSF', 'OverallCond', 
                                         'OverallQual', 'TotalBsmtSF', 
                                         'YearBuilt', 'YearRemodAdd'])),

        ("winsorizer_iqr", Winsorizer(capping_method='iqr',
                            tail='both', fold=1.5, 
                            variables= ['1stFlrSF', '2ndFlrSF', 'BsmtFinSF1',
        'BsmtUnfSF', 'GarageArea', 'GarageYrBlt', 'GrLivArea', 'LotArea',
        'LotFrontage', 'MasVnrArea', 'OpenPorchSF', 'TotalBsmtSF', 'YearBuilt',
        'YearRemodAdd'])),

        ("OrdinalCategoricalEncoder", OrdinalEncoder(encoding_method='arbitrary', 
        variables=['BsmtExposure', 'BsmtFinType1', 'GarageFinish', 
        'KitchenQual'])),
        
        ("yjt", vt.YeoJohnsonTransformer(variables = ['1stFlrSF', '2ndFlrSF',
        'BedroomAbvGr', 'BsmtFinSF1', 'BsmtUnfSF', 'GarageArea', 'GarageYrBlt',
        'GrLivArea', 'LotArea', 'LotFrontage', 'MasVnrArea', 'OpenPorchSF',
        'OverallCond', 'OverallQual', 'TotalBsmtSF', 'YearBuilt', 'YearRemodAdd', 
        'BsmtExposure', 'BsmtFinType1', 'GarageFinish', 
        'KitchenQual'])),

        ("SmartCorrelatedSelection", SmartCorrelatedSelection(variables=None,
         method="spearman", threshold=0.6, selection_method="variance")),

        ("feat_scaling", StandardScaler()),

        ("feat_selection",  SelectFromModel(model)),

        ("model", model),

    ])

    return pipeline_base

# Split Train & Test Set

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(['SalePrice'], axis=1),
    df['SalePrice'],
    test_size=0.2,
    random_state=0
)

print("* Train set:", X_train.shape, y_train.shape,
      "\n* Test set:",  X_test.shape, y_test.shape)

In [ ]:
y_test.head()

# Grid Search CV - Sklearn

* Use default hyperparameters to find most suitable algorithm

In [ ]:
models_quick_search = {
    'LinearRegression': LinearRegression(),
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=0),
    "RandomForestRegressor": RandomForestRegressor(random_state=0),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
    "AdaBoostRegressor": AdaBoostRegressor(random_state=0),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=0),
    "XGBRegressor": XGBRegressor(random_state=0),
}

params_quick_search = {
    'LinearRegression': {},
    "DecisionTreeRegressor": {},
    "RandomForestRegressor": {},
    "ExtraTreesRegressor": {},
    "AdaBoostRegressor": {},
    "GradientBoostingRegressor": {},
    "XGBRegressor": {},
}

* Load Custom Class for hyperparameter optimisation (provided by Code Institute)

In [ ]:
from sklearn.model_selection import GridSearchCV


class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")
            model = PipelineOptimization(self.models[key])

            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring)
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)

        columns = ['estimator', 'min_score',
                   'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]

        return df[columns], self.grid_searches

* Perform a hyperparameter optimisation search using the default hyperparameters

In [ ]:
search = HyperparameterOptimizationSearch(models=models_quick_search, params=params_quick_search)
search.fit(X_train, y_train, scoring='r2', n_jobs=-1, cv=5)

* Check the results

In [ ]:
import numpy as np
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
grid_search_summary

# Perform an extensive search on the most suitable model to find the best hyperparameter configuration

#### Define model and parameters, for Extensive Search

In [ ]:
models_search = {
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=0),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
}

params_search = {
    "GradientBoostingRegressor": {
        'model__n_estimators': [100,300],
        'model__learning_rate': [1e-1,1e-2,1e-3], 
        'model__max_depth': [3,10,None],
        'model__min_samples_split': [2,50],
        'model__min_samples_leaf': [1,50],
        'model__max_leaf_nodes': [None,50],

    },

    "ExtraTreesRegressor": {
        'model__n_estimators': [100, 50, 150],
        'model__max_depth': [None, 3, 15],
        'model__min_samples_split': [2, 50],
        'model__min_samples_leaf': [1, 50],
        'model__max_features': [1.0, 'sqrt', 'log2'],
        'model__bootstrap': [True, False],
    }

}

#### Extensive GridSearch CV


In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train, y_train, scoring = 'r2', n_jobs=-1, cv=5)

#### Check the results

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
grid_search_summary

#### Get the best model

In [ ]:
best_model = grid_search_summary.iloc[0, 0]
best_model

#### Get the parameters for best model

In [ ]:
best_parameters = grid_search_pipelines[best_model].best_params_
best_parameters

#### Define the best regressor, based on search

In [ ]:
best_regressor_pipeline = grid_search_pipelines[best_model].best_estimator_
best_regressor_pipeline

# Evaluate on Train and Test Sets

* Use custom functions defined by Code Institute

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
sns.set_style('whitegrid')
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


def regression_performance(X_train, y_train, X_test, y_test, pipeline):
    print("Model Evaluation \n")
    print("* Train Set")
    regression_evaluation(X_train, y_train, pipeline)
    print("* Test Set")
    regression_evaluation(X_test, y_test, pipeline)


def regression_evaluation(X, y, pipeline):
    prediction = pipeline.predict(X)
    print('R2 Score:', r2_score(y, prediction).round(3))
    print('Mean Absolute Error:', mean_absolute_error(y, prediction).round(3))
    print('Mean Squared Error:', mean_squared_error(y, prediction).round(3))
    print('Root Mean Squared Error:', np.sqrt(
        mean_squared_error(y, prediction)).round(3))
    print("\n")


def regression_evaluation_plots(X_train, y_train, X_test, y_test, pipeline, alpha_scatter=0.5):
    pred_train = pipeline.predict(X_train)
    pred_test = pipeline.predict(X_test)

    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))
    sns.scatterplot(x=y_train, y=pred_train, alpha=alpha_scatter, ax=axes[0])
    sns.lineplot(x=y_train, y=y_train, color='red', ax=axes[0])
    axes[0].set_xlabel("Actual")
    axes[0].set_ylabel("Predictions")
    axes[0].set_title("Train Set")

    sns.scatterplot(x=y_test, y=pred_test, alpha=alpha_scatter, ax=axes[1])
    sns.lineplot(x=y_test, y=y_test, color='red', ax=axes[1])
    axes[1].set_xlabel("Actual")
    axes[1].set_ylabel("Predictions")
    axes[1].set_title("Test Set")

    plt.show()

#### Evaluate Performance

In [ ]:
regression_performance(X_train, y_train, X_test, y_test, best_regressor_pipeline)
regression_evaluation_plots(X_train, y_train, X_test, y_test, best_regressor_pipeline)

# Assess feature importance

In [ ]:
# how many data cleaning and feature engineering steps does your pipeline have?
data_cleaning_feat_eng_steps = 6
columns_after_data_cleaning_feat_eng = (Pipeline(best_regressor_pipeline.steps[:data_cleaning_feat_eng_steps])
                                        .transform(X_train)
                                        .columns)

best_features = columns_after_data_cleaning_feat_eng[best_regressor_pipeline['feat_selection'].get_support(
)].to_list()

# create DataFrame to display feature importance
df_feature_importance = (pd.DataFrame(data={
    'Feature': columns_after_data_cleaning_feat_eng[best_regressor_pipeline['feat_selection'].get_support()],
    'Importance': best_regressor_pipeline['model'].feature_importances_})
    .sort_values(by='Importance', ascending=False)
)

# Most important features statement and plot
print(f"* These are the {len(best_features)} most important features in descending order. "
      f"The model was trained on them: \n{df_feature_importance['Feature'].to_list()}")

df_feature_importance.plot(kind='bar', x='Feature', y='Importance')
plt.show()

# Refit pipeline with best features

* Note that all features are numerical variables. Therefore:
  * No need for categorical imputation.
  * While OverallQual is represented numerically, its discrete, ordinal nature 
    means Winsorizer is not applied, as it's designed for continuous outliers.

In [ ]:
best_features = df_feature_importance['Feature'].to_list()
best_features

#### Rewrite the Regressor Pipeline

In [ ]:
def PipelineOptimization(model):
    pipeline_base = Pipeline([

        ("numerical_imputation", MeanMedianImputer(imputation_method='median',
                            variables= ['OverallQual', 
                                         'GarageArea', 
                                         'TotalBsmtSF', 
                                         'YearRemodAdd'])),

        ("winsorizer_iqr", Winsorizer(capping_method='iqr',
                            tail='both', fold=1.5, 
                            variables= ['GarageArea','TotalBsmtSF'])),
        
        ("yjt", vt.YeoJohnsonTransformer(variables = ['OverallQual', 
                                                      'GarageArea', 
                                                      'TotalBsmtSF', 
                                                      'YearRemodAdd'])),

        ("feat_scaling", StandardScaler()),

        ("model", model),

    ])

    return pipeline_base

# Split Train & Test Set, only with best features

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(['SalePrice'], axis=1),
    df['SalePrice'],
    test_size=0.2,
    random_state=0
)

print("* Train set:", X_train.shape, y_train.shape,
      "\n* Test set:",  X_test.shape, y_test.shape)

#### Subset Best Features

In [ ]:
X_train = X_train.filter(best_features)
X_test = X_test.filter(best_features)

print("* Train set:", X_train.shape, y_train.shape, "\n* Test set:",  X_test.shape, y_test.shape)
X_test.head()

# Grid Search CV – Sklearn

* Use the same model from the previous GridCV search

In [ ]:
models_search


In [ ]:
best_model_after_search = {
    'ExtraTreesRegressor': models_search['ExtraTreesRegressor']
}
best_model_after_search

* Get the best parameters of that model

In [ ]:
best_parameters


In [ ]:
params_search = {'ExtraTreesRegressor':  {
    'model__bootstrap': [True],
    'model__max_depth': [15],
    'model__max_features': ['sqrt'],
    'model__min_samples_leaf': [1],
    'model__min_samples_split': [2],
    'model__n_estimators': [100]
}
}

params_search

* GridSearch CV

In [ ]:
search = HyperparameterOptimizationSearch(models=best_model_after_search, params=params_search)
search.fit(X_train, y_train, scoring = 'r2', n_jobs=-1, cv=5)

* Check the results

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
grid_search_summary

* Check the best model

In [ ]:
best_model = grid_search_summary.iloc[0,0]
best_model

* Define the best regressor pipeline

In [ ]:
best_regressor_pipeline= grid_search_pipelines[best_model].best_estimator_
best_regressor_pipeline

# Assess feature importance

In [ ]:
# how many data cleaning and feature engineering does your pipeline have?
data_cleaning_feat_eng_steps = 3
columns_after_data_cleaning_feat_eng = (Pipeline(best_regressor_pipeline.steps[:data_cleaning_feat_eng_steps])
                                        .transform(X_train)
                                        .columns)

best_features = columns_after_data_cleaning_feat_eng

# create DataFrame to display feature importance
df_feature_importance = (pd.DataFrame(data={
    'Feature': columns_after_data_cleaning_feat_eng,
    'Importance': best_regressor_pipeline['model'].feature_importances_})
    .sort_values(by='Importance', ascending=False)
)

# Most important features statement and plot
print(f"* These are the {len(best_features)} most important features in descending order. "
      f"The model was trained on them: \n{df_feature_importance['Feature'].to_list()}")

df_feature_importance.plot(kind='bar', x='Feature', y='Importance')
plt.show()

# Evaluate Regressor Pipeline on Train and Test Sets

In [ ]:
regression_performance(X_train, y_train, X_test, y_test, best_regressor_pipeline)
regression_evaluation_plots(X_train, y_train, X_test, y_test, best_regressor_pipeline)

---

# Push files to Repo

In [ ]:
import joblib
import os

version = 'v1'
file_path = f'outputs/ml_pipeline/predict_saleprice/{version}'

try:
  os.makedirs(name=file_path)
except Exception as e:
  print(e)

### Save Train Set

In [ ]:
X_train.head()

In [ ]:
X_train.to_csv(f"{file_path}/X_train.csv", index=False)

In [ ]:
y_train

In [ ]:
y_train.to_csv(f"{file_path}/y_train.csv", index=False)

### Save Test Set

In [ ]:
X_test.head()

In [ ]:
X_test.to_csv(f"{file_path}/X_test.csv", index=False)

In [ ]:
y_test

In [ ]:
y_test.to_csv(f"{file_path}/y_test.csv", index=False)

### Save Modelling pipeline for predicting Sales Price

In [ ]:
best_regressor_pipeline

In [ ]:
joblib.dump(value=best_regressor_pipeline, filename=f"{file_path}/best_features_regressor_pipeline.pkl")

### Save feature importance plot

In [ ]:
df_feature_importance.plot(kind='bar', x='Feature', y='Importance')
plt.show()

In [ ]:
df_feature_importance.plot(kind='bar', x='Feature', y='Importance')
plt.savefig(f'{file_path}/features_importance.png', bbox_inches='tight')